
**Grupo 11** · Hospital Regional del Altiplano · UCI como cola M/M/c y M/M/c/c.

Parámetros: $\lambda = 3.0+0.2((g-1)\bmod 5)=3.0$, $c=17+((g-1)\bmod 3)=18$, $\mu=0.25$, $a=\lambda/\mu=12$, $\rho=a/c=2/3$,
$k_{off}=\lceil c-a\rceil+3=9$, $T_b=7$ días.

In [1]:
import math, heapq, json, importlib.util, types, sys, os
from collections import deque
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
os.makedirs("figs", exist_ok=True)

G = 11
LAM = 3.0 + 0.2*((G-1) % 5)
C   = 17 + ((G-1) % 3)
MU  = 0.25
A   = LAM/MU
RHO = A/C
TB  = 7.0
KOFF = math.ceil(C-A) + 3
print(dict(g=G, lam=LAM, c=C, mu=MU, a=A, rho=round(RHO,4), k_off=KOFF, Tb=TB))
R = {}   # resultados que luego se vuelcan al informe
R["params"] = dict(g=G, lam=LAM, c=C, mu=MU, a=A, rho=RHO, k_off=KOFF, Tb=TB)

{'g': 11, 'lam': 3.0, 'c': 18, 'mu': 0.25, 'a': 12.0, 'rho': 0.6667, 'k_off': 9, 'Tb': 7.0}


## Task 1.1 - Teoría
Fórmulas (diapositivas de la sesión de colas):

* Carga ofrecida $a=\lambda/\mu$, utilización $\rho=a/c$.
* **Erlang B** (recurrencia): $B(0,a)=1,\; B(k,a)=\dfrac{a\,B(k-1,a)}{k+a\,B(k-1,a)}$.
* **Erlang C**: $C(c,a)=\dfrac{B(c,a)}{1-\rho\,(1-B(c,a))}$ (probabilidad de esperar en M/M/c, $\rho<1$).
* $L_q=C(c,a)\dfrac{\rho}{1-\rho}$, $W_q=L_q/\lambda$, $W=W_q+1/\mu$, $L=L_q+a$.
* M/M/c/c: bloqueo $B(c,a)$, tasa efectiva $\lambda_{ef}=\lambda(1-B)$, camas ocupadas $L=\lambda_{ef}/\mu=a(1-B)$, $W=1/\mu$.
* Little: $L=\lambda_{(ef)}W$, $L_q=\lambda W_q$.

In [2]:
def erlang_b(c, a):
    # Recurrencia de Erlang B: B(k) = a B(k-1) / (k + a B(k-1))
    b = 1.0
    for k in range(1, c+1):
        b = a*b/(k + a*b)
    return b

def erlang_c(c, a):
    # Erlang C a partir de Erlang B: C = B / (1 - rho (1 - B))
    rho = a/c; b = erlang_b(c, a)
    return b/(1 - rho*(1 - b))

def teoria_mmc(lam, mu, c):
    a = lam/mu; rho = a/c
    Cw = erlang_c(c, a)
    Lq = Cw*rho/(1-rho); Wq = Lq/lam; W = Wq + 1/mu; L = Lq + a
    return dict(a=a, rho=rho, C=Cw, Lq=Lq, Wq_d=Wq, Wq_h=Wq*24, W_d=W, W_h=W*24, L=L)

def teoria_mmcc(lam, mu, c):
    a = lam/mu; B = erlang_b(c, a)
    lam_ef = lam*(1-B); L = lam_ef/mu
    return dict(a=a, B=B, lam_ef=lam_ef, L=L, util=L/c, W_d=1/mu, W_h=24/mu)

TW = teoria_mmc(LAM, MU, C); TS = teoria_mmcc(LAM, MU, C)
print("M/M/c   :", {k: round(v,5) for k,v in TW.items()})
print("M/M/c/c :", {k: round(v,5) for k,v in TS.items()})
# Little (teórico)
print("\nLittle M/M/c   : Lq - lam*Wq =", TW['Lq']-LAM*TW['Wq_d'], "| L - lam*W =", TW['L']-LAM*TW['W_d'])
print("Little M/M/c/c : L - lam_ef*W =", TS['L']-TS['lam_ef']*TS['W_d'])
print("Error si se usa lam en lugar de lam_ef: L=lam*W =", LAM*TS['W_d'], "vs", TS['L'],
      f"-> sobreestima {100*(LAM*TS['W_d']/TS['L']-1):.2f} %")
R["teoria_mmc"] = TW; R["teoria_mmcc"] = TS
R["little_lam_error_pct"] = 100*(LAM*TS['W_d']/TS['L']-1)

M/M/c   : {'a': 12.0, 'rho': 0.66667, 'C': 0.07561, 'Lq': 0.15123, 'Wq_d': 0.05041, 'Wq_h': 1.20984, 'W_d': 4.05041, 'W_h': 97.20984, 'L': 12.15123}
M/M/c/c : {'a': 12.0, 'B': 0.02654, 'lam_ef': 2.92037, 'L': 11.68148, 'util': 0.64897, 'W_d': 4.0, 'W_h': 96.0}

Little M/M/c   : Lq - lam*Wq = 0.0 | L - lam*W = 0.0
Little M/M/c/c : L - lam_ef*W = 0.0
Error si se usa lam en lugar de lam_ef: L=lam*W = 12.0 vs 11.681484585976623 -> sobreestima 2.73 %


## Task 1.2 – Simulador original generado por IA (sin modificar)
El prompt y el código **tal como se recibió** están en `anexo/` (`prompt_task_1_2.txt`, `simulador_original_ia.py`).
Aquí solo lo cargamos y lo ejecutamos tal cual. Las afirmaciones del agente sobre su propio trabajo se copian literalmente en el informe (anexo B).

In [3]:
def cargar_original():
    spec = importlib.util.spec_from_file_location("orig", "anexo/simulador_original_ia.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
orig = cargar_original()
for conesp in (True, False):
    r = orig.replicas(LAM, MU, C, 1000, 100, 30, conesp)
    print("con espera" if conesp else "sin espera")
    for k, v in r.items(): print(f"   {k:12s} media={v[0]:.4f}  IC=({v[1]:.4f}, {v[2]:.4f})")

con espera
   L            media=12.4650  IC=(12.4650, 12.4650)
   Lq           media=0.1203  IC=(0.1203, 0.1203)
   Wq           media=0.0323  IC=(0.0323, 0.0323)
   utilizacion  media=0.6593  IC=(0.6593, 0.6593)
   P_esperar    media=0.0644  IC=(0.0644, 0.0644)
   P_bloqueo    media=0.0000  IC=(0.0000, 0.0000)
   llegadas     media=2687.0000  IC=(2687.0000, 2687.0000)
sin espera
   L            media=11.9599  IC=(11.9599, 11.9599)
   Lq           media=0.0000  IC=(0.0000, 0.0000)
   Wq           media=0.0000  IC=(0.0000, 0.0000)
   utilizacion  media=0.6432  IC=(0.6432, 0.6432)
   P_esperar    media=0.0000  IC=(0.0000, 0.0000)
   P_bloqueo    media=0.0208  IC=(0.0208, 0.0208)
   llegadas     media=2688.0000  IC=(2688.0000, 2688.0000)


## Task 1.3 – Simulador corregido
Correcciones respecto al original (puntos i–vii):
* (i) $L$, $L_q$ y utilización se integran **ponderando por la duración** de cada tramo: $\bar L=\frac1T\int n(t)\,dt$.
* (ii) el IC se calcula sobre **medias por réplica** con $t_{n-1}$ (no $z$).
* (iii) calentamiento $t_w$ justificado (Task 1.4) y las entidades se clasifican **por su instante de llegada** ($\ge t_w$).
* (iv) cada réplica usa un `SeedSequence.spawn` independiente.
* (v) se reporta $\lambda_{ef}$ = admitidos/T y todo lo "por paciente" usa admitidos.
* (vi) al llegar al horizonte se cierran las integrales exactamente en $T_h$ y los pacientes aún en fila cuentan su espera parcial (censurada).
* (vii) la espera se mide desde la **llegada**.

Extensiones para el Task 2: distribución de estancia (`exp`, `det`, `lognorm`), cierre parcial de camas (`cierre=(t0,Tb,k)`), rejilla temporal de $L_q(t)$.

In [4]:
ARR, DEP, CLOSE, OPEN = 0, 1, 2, 3

def sampler_estancia(rng, mu, dist, cs):
    m = 1.0/mu
    if dist == "exp":   return lambda: rng.exponential(m)
    if dist == "det":   return lambda: m
    if dist == "lognorm":                    # s^2 = ln(1+Cs^2), m_ln = ln(1/mu) - s^2/2
        s2 = math.log(1 + cs**2); ml = math.log(m) - s2/2; s = math.sqrt(s2)
        return lambda: rng.lognormal(ml, s)
    raise ValueError(dist)

def simular(lam, mu, c, horizonte, calentamiento=0.0, semilla=None, con_espera=True,
            dist="exp", cs=1.0, arr="exp", cierre=None, grid_dt=None):
    # DES con lista de eventos (heap). Estado: ocupadas, fila FIFO. cierre=(t0, Tb, k_off).
    ss = semilla if isinstance(semilla, np.random.SeedSequence) else np.random.SeedSequence(semilla)
    rng = np.random.default_rng(ss)
    serv = sampler_estancia(rng, mu, dist, cs)
    inter = (lambda: rng.exponential(1.0/lam)) if arr == "exp" else (lambda: 1.0/lam)
    ev = []; seq = 0
    def push(t, tipo, dato=None):
        nonlocal seq; heapq.heappush(ev, (t, seq, tipo, dato)); seq += 1
    push(inter(), ARR)
    cap = c
    if cierre: push(cierre[0], CLOSE); push(cierre[0]+cierre[1], OPEN)
    busy = 0; fila = deque(); t = 0.0
    aL = aLq = aBusy = 0.0                       # integrales ponderadas por duración
    n_arr = n_blk = n_adm = n_wait = 0
    sum_wq = 0.0; soj = []
    if grid_dt:
        grid = np.arange(0.0, horizonte, grid_dt); gL = np.zeros(len(grid)); gLq = np.zeros(len(grid)); gi = 0
    def avanzar(t_new):
        nonlocal t, aL, aLq, aBusy, gi
        if grid_dt:
            while gi < len(grid) and grid[gi] < t_new:    # estado vigente en [t, t_new)
                gL[gi] = busy+len(fila); gLq[gi] = len(fila); gi += 1
        lo = max(t, calentamiento); hi = min(t_new, horizonte)
        if hi > lo:
            d = hi - lo; aL += (busy+len(fila))*d; aLq += len(fila)*d; aBusy += busy*d
        t = t_new
    def iniciar(t_llegada):
        nonlocal busy, sum_wq, n_wait
        busy += 1; push(t + serv(), DEP, t_llegada)
        if t_llegada >= calentamiento:                    # (vii) espera desde la llegada; (iii) por instante de llegada
            w = t - t_llegada; sum_wq += w; n_wait += (w > 0)
    while ev:
        te, _, tipo, dato = heapq.heappop(ev)
        if te > horizonte: break
        avanzar(te)
        if tipo == ARR:
            push(t + inter(), ARR)
            ven = t >= calentamiento
            if ven: n_arr += 1
            if busy < cap and not fila:
                if ven: n_adm += 1
                iniciar(t)
            elif con_espera:
                if ven: n_adm += 1
                fila.append(t)
            elif ven: n_blk += 1
        elif tipo == DEP:
            busy -= 1
            if dato >= calentamiento: soj.append(t - dato)
            while fila and busy < cap: iniciar(fila.popleft())
        elif tipo == CLOSE: cap = c - cierre[2]
        elif tipo == OPEN:
            cap = c
            while fila and busy < cap: iniciar(fila.popleft())
    avanzar(horizonte)                                    # (vi) cierra la integral exactamente en el horizonte
    if grid_dt:
        while gi < len(grid): gL[gi] = busy+len(fila); gLq[gi] = len(fila); gi += 1
    for ta in fila:                                       # (vi) espera parcial de quienes siguen en fila
        if ta >= calentamiento:
            w = horizonte - ta; sum_wq += w; n_wait += (w > 0)
    T = horizonte - calentamiento
    out = dict(L=aL/T, Lq=aLq/T, util=aBusy/(T*c), Wq_d=sum_wq/n_adm if n_adm else 0.0,
               W_d=float(np.mean(soj)) if soj else float("nan"),
               P_esperar=n_wait/n_adm if n_adm else 0.0, P_bloqueo=n_blk/n_arr if n_arr else 0.0,
               lam_ef=n_adm/T, n_arr=n_arr)
    if grid_dt: out["grid"] = grid; out["gL"] = gL; out["gLq"] = gLq
    return out

def replicas(n, semilla_base, **kw):
    # (iv) semillas independientes: SeedSequence.spawn
    return [simular(semilla=s, **kw) for s in np.random.SeedSequence(semilla_base).spawn(n)]

def ic95(x):
    x = np.asarray(x, float); n = len(x); m = x.mean(); s = x.std(ddof=1)
    h = stats.t.ppf(0.975, n-1)*s/math.sqrt(n); return m, m-h, m+h        # (ii) la unidad es la réplica

### Task 1.3b – Pruebas diseñadas con respuesta conocida
Para ejecutar el código original con entradas **deterministas** se sustituye temporalmente `random.Random` por un generador falso cuyo `expovariate(r)` devuelve `1/r`.

In [5]:
class FakeRandom:
    def __init__(self, seed=None): pass
    def expovariate(self, rate): return 1.0/rate

def orig_det(**kw):
    real = orig.random
    orig.random = types.SimpleNamespace(Random=FakeRandom)
    try: return orig.simular(**kw)
    finally: orig.random = real

pruebas = []
# --- Prueba A: D/D/1, llegadas en t=1,2,...,100 (cada 1 día), servicio 0.25 días, 1 cama, T=100.5.
# Respuesta exacta a mano: 100 servicios x 0.25 = 25 camas-día -> L = util = 25/100.5 = 0.24876, Lq = 0, Wq = 0
kwA = dict(lam=1.0, mu=4.0, c=1, horizonte=100.5, calentamiento=0.0)
oA = orig_det(**kwA, con_espera=True)
cA = simular(**kwA, semilla=1, arr="det", dist="det")
EXA = 25/100.5
pruebas.append(("A  D/D/1 (L exacto 25/100.5)", "L", EXA, oA["L"], cA["L"]))
pruebas.append(("A  D/D/1 (util exacta 25/100.5)", "utilizacion", EXA, oA["utilizacion"], cA["util"]))
# --- Prueba A2: mismo caso con horizonte 100.1 que corta el último servicio: 99 servicios completos + 0.1 -> util = (99*0.25+0.1)/100.1
kwA2 = dict(lam=1.0, mu=4.0, c=1, horizonte=100.1, calentamiento=0.0)
oA2 = orig_det(**kwA2, con_espera=True); cA2 = simular(**kwA2, semilla=1, arr="det", dist="det")
ex = (99*0.25 + 0.1)/100.1
pruebas.append(("A2 horizonte corta servicio (util exacta)", "utilizacion", ex, oA2["utilizacion"], cA2["util"]))
# --- Prueba B: M/M/1/1 (1 cama, sin espera) lam=mu=1 -> B=0.5, L=0.5, util=0.5, lam_ef=0.5
kwB = dict(lam=1.0, mu=1.0, c=1, horizonte=40000.0, calentamiento=100.0)
oB = orig.simular(**kwB, semilla=7, con_espera=False); cB = simular(**kwB, semilla=7, con_espera=False)
pruebas.append(("B  M/M/1/1 (L teórico 0.5)", "L", 0.5, oB["L"], cB["L"]))
pruebas.append(("B  M/M/1/1 (P_bloqueo 0.5)", "P_bloqueo", 0.5, oB["P_bloqueo"], cB["P_bloqueo"]))
# --- Prueba C: M/M/1 lam=0.5 mu=1 -> L=1, Lq=0.5, Wq=1, P_esperar=0.5
kwC = dict(lam=0.5, mu=1.0, c=1, horizonte=40000.0, calentamiento=100.0)
oC = orig.simular(**kwC, semilla=7, con_espera=True); cC = simular(**kwC, semilla=7, con_espera=True)
pruebas.append(("C  M/M/1 (L teórico 1)", "L", 1.0, oC["L"], cC["L"]))
pruebas.append(("C  M/M/1 (Lq teórico 0.5)", "Lq", 0.5, oC["Lq"], cC["Lq"]))
pruebas.append(("C  M/M/1 (Wq teórico 1 día)", "Wq", 1.0, oC["Wq"], cC["Wq_d"]))
# --- Prueba D: sobrecarga determinista (lam=2, mu=1, c=1), calentamiento 10, horizonte 110: toda llegada de la ventana espera -> P_esperar ~ 1
kwD = dict(lam=2.0, mu=1.0, c=1, horizonte=110.0, calentamiento=10.0)
oD = orig_det(**kwD, con_espera=True); cD = simular(**kwD, semilla=1, arr="det", dist="det")
pruebas.append(("D  sobrecarga (P_esperar ~1)", "P_esperar", 1.0, oD["P_esperar"], cD["P_esperar"]))
T13 = pd.DataFrame(pruebas, columns=["prueba","medida","valor correcto","original","corregido"])
T13["err_orig_%"] = 100*(T13["original"]/T13["valor correcto"]-1)
T13["err_corr_%"] = 100*(T13["corregido"]/T13["valor correcto"]-1)
print(T13.round(4).to_string(index=False))
R["pruebas_1_3"] = T13.round(5).to_dict("records")
# --- Prueba E: semillas (punto iv) e IC (ii)
rO = [orig.simular(LAM, MU, C, 1000, 100, 42, True)["Wq"] for _ in range(5)]
print("\nWq de 5 'réplicas' del original (semilla fija 42):", rO)
R["prueba_semilla_original"] = rO

                                   prueba      medida  valor correcto  original  corregido  err_orig_%  err_corr_%
             A  D/D/1 (L exacto 25/100.5)           L          0.2488    0.5000     0.2488    101.0000      0.0000
          A  D/D/1 (util exacta 25/100.5) utilizacion          0.2488    0.2488     0.2488      0.0000      0.0000
A2 horizonte corta servicio (util exacta) utilizacion          0.2483    0.2473     0.2483     -0.4024     -0.0000
               B  M/M/1/1 (L teórico 0.5)           L          0.5000    0.6684     0.4980     33.6799     -0.4059
               B  M/M/1/1 (P_bloqueo 0.5)   P_bloqueo          0.5000    0.5039     0.4999      0.7727     -0.0226
                   C  M/M/1 (L teórico 1)           L          1.0000    1.4800     1.0070     47.9971      0.7014
                C  M/M/1 (Lq teórico 0.5)          Lq          0.5000    0.7302     0.5058     46.0468      1.1537
              C  M/M/1 (Wq teórico 1 día)          Wq          1.0000    0.9740 

## Task 1.4 – Validación contra la teoría
**(a) Calentamiento y horizonte.** Se promedian 50 réplicas arrancando con el hospital vacío y se grafica el promedio móvil de la fila $L_q(t)$ y del número en el sistema $L(t)$.

In [6]:
rep0 = replicas(50, 2024, lam=LAM, mu=MU, c=C, horizonte=200.0, calentamiento=0.0, con_espera=True, grid_dt=0.5)
g = rep0[0]["grid"]
mLq = np.mean([r["gLq"] for r in rep0], axis=0); mL = np.mean([r["gL"] for r in rep0], axis=0)
def movavg(x, w): return np.convolve(x, np.ones(w)/w, mode="valid")
w = 10                                                  # ventana de 5 días
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(g, mL, alpha=.35, label="promedio de 50 réplicas"); ax[0].plot(g[w-1:], movavg(mL, w), lw=2, label="promedio móvil (5 d)")
ax[0].axhline(TW["L"], color="k", ls="--", label=f"teoría L={TW['L']:.2f}"); ax[0].axvline(40, color="r", ls=":", label="calentamiento = 40 d")
ax[0].set_xlabel("t (días)"); ax[0].set_ylabel("L(t)"); ax[0].set_title("Número en el sistema"); ax[0].legend(fontsize=8)
ax[1].plot(g, mLq, alpha=.35); ax[1].plot(g[w-1:], movavg(mLq, w), lw=2, label="promedio móvil (5 d)")
ax[1].axhline(TW["Lq"], color="k", ls="--", label=f"teoría Lq={TW['Lq']:.3f}"); ax[1].axvline(40, color="r", ls=":")
ax[1].set_xlabel("t (días)"); ax[1].set_ylabel("Lq(t)"); ax[1].set_title("Longitud de la fila"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("figs/fig_calentamiento.png", dpi=140); plt.close()
ma = movavg(mL, w); t_ma = g[w-1:]
idx = np.where(np.abs(ma/TW["L"]-1) < 0.02)[0][0]
print("el promedio móvil de L(t) entra a ±2 % de la teoría en t ≈", t_ma[idx], "d")
print("L(t)=a(1-e^{-mu t}): error < 1 % para t >", math.log(100)/MU, "d  (5/mu ≈", 5/MU, "d)")

el promedio móvil de L(t) entra a ±2 % de la teoría en t ≈ 13.5 d
L(t)=a(1-e^{-mu t}): error < 1 % para t > 18.420680743952367 d  (5/mu ≈ 20.0 d)


In [7]:
WARM, HOR, NREP = 40.0, 3000.0, 50       # calentamiento 40 d (≈10 estancias medias), 3000 d de horizonte, 50 réplicas

def tabla_validacion(reps, teo):
    filas = []
    for clave, nombre, esc in [("L","L",1),("Lq","Lq",1),("Wq_d","Wq (h)",24),("W_d","W (h)",24),
                               ("util","utilización",1),("P_esperar","P(esperar)",1),("P_bloqueo","P(bloqueo)",1),("lam_ef","λ_ef (pac/día)",1)]:
        m, lo, hi = ic95([r[clave]*esc for r in reps])
        t = teo.get(nombre)
        filas.append(dict(medida=nombre, media=m, IC_inf=lo, IC_sup=hi, teoria=t,
                          dentro=("sí" if (t is not None and lo <= t <= hi) else "no")))
    return pd.DataFrame(filas)

teo_w = {"L":TW["L"],"Lq":TW["Lq"],"Wq (h)":TW["Wq_h"],"W (h)":TW["W_h"],"utilización":TW["rho"],
         "P(esperar)":TW["C"],"P(bloqueo)":0.0,"λ_ef (pac/día)":LAM}
teo_s = {"L":TS["L"],"Lq":0.0,"Wq (h)":0.0,"W (h)":TS["W_h"],"utilización":TS["util"],
         "P(esperar)":0.0,"P(bloqueo)":TS["B"],"λ_ef (pac/día)":TS["lam_ef"]}
repW = replicas(NREP, 11, lam=LAM, mu=MU, c=C, horizonte=HOR, calentamiento=WARM, con_espera=True)
repS = replicas(NREP, 12, lam=LAM, mu=MU, c=C, horizonte=HOR, calentamiento=WARM, con_espera=False)
VW = tabla_validacion(repW, teo_w); VS = tabla_validacion(repS, teo_s)
print("VARIANTE CON ESPERA (M/M/18), 50 réplicas"); print(VW.round(5).to_string(index=False))
print("\nVARIANTE SIN ESPERA (M/M/18/18), 50 réplicas"); print(VS.round(5).to_string(index=False))
R["val_w"] = VW.round(6).to_dict("records"); R["val_s"] = VS.round(6).to_dict("records")
R["val_cfg"] = dict(warm=WARM, hor=HOR, nrep=NREP)

VARIANTE CON ESPERA (M/M/18), 50 réplicas
        medida    media   IC_inf   IC_sup   teoria dentro
             L 12.14426 12.07784 12.21067 12.15123     sí
            Lq  0.14414  0.12980  0.15849  0.15123     sí
        Wq (h)  1.14868  1.03625  1.26111  1.20984     sí
         W (h) 96.87406 96.49362 97.25449 97.20984     sí
   utilización  0.66667  0.66346  0.66988  0.66667     sí
    P(esperar)  0.07449  0.07091  0.07806  0.07561     sí
    P(bloqueo)  0.00000  0.00000  0.00000  0.00000     sí
λ_ef (pac/día)  3.00457  2.99460  3.01455  3.00000     sí

VARIANTE SIN ESPERA (M/M/18/18), 50 réplicas
        medida    media   IC_inf   IC_sup   teoria dentro
             L 11.66394 11.61962 11.70826 11.68148     sí
            Lq  0.00000  0.00000  0.00000  0.00000     sí
        Wq (h)  0.00000  0.00000  0.00000  0.00000     sí
         W (h) 95.92131 95.63731 96.20532 96.00000     sí
   utilización  0.64800  0.64553  0.65046  0.64897     sí
    P(esperar)  0.00000  0.00000  0.00000 

**(b) Ley de Little con las cantidades de la simulación** (independiente de la teoría): $L$ viene de la integral en el tiempo, $\lambda_{ef}$ de los admitidos, $W$ de los tiempos individuales.

In [8]:
def little(reps):
    L = np.array([r["L"] for r in reps]); l = np.array([r["lam_ef"] for r in reps]); W = np.array([r["W_d"] for r in reps])
    err = np.abs(L - l*W)/L
    return dict(L=L.mean(), lam_ef=l.mean(), W_d=W.mean(), L_pred=(l*W).mean(),
                err_rel_medio_pct=100*err.mean(), err_rel_de_medias_pct=100*abs(L.mean()-l.mean()*W.mean())/L.mean())
LW = little(repW); LS = little(repS)
print("con espera:", {k: round(v,5) for k,v in LW.items()}); print("sin espera:", {k: round(v,5) for k,v in LS.items()})
R["little_w"] = LW; R["little_s"] = LS

con espera: {'L': np.float64(12.14426), 'lam_ef': np.float64(3.00457), 'W_d': np.float64(4.03642), 'L_pred': np.float64(12.12795), 'err_rel_medio_pct': np.float64(0.13404), 'err_rel_de_medias_pct': np.float64(0.13616)}
sin espera: {'L': np.float64(11.66394), 'lam_ef': np.float64(2.91476), 'W_d': np.float64(3.99672), 'L_pred': np.float64(11.64932), 'err_rel_medio_pct': np.float64(0.12543), 'err_rel_de_medias_pct': np.float64(0.12381)}


**(c) Afirmaciones del agente vs. verificación.** Se contrasta cada afirmación con lo medido en las pruebas del Task 1.3 y en la validación.

In [9]:
rv = orig.replicas(LAM, MU, C, 1000, 100, 30, True); rv_s = orig.replicas(LAM, MU, C, 1000, 100, 30, False)
filas = [
 ("L (con espera)", rv["L"][0], TW["L"], rv["L"][1], rv["L"][2]),
 ("Wq horas (con espera)", rv["Wq"][0]*24, TW["Wq_h"], rv["Wq"][1]*24, rv["Wq"][2]*24),
 ("P(esperar)", rv["P_esperar"][0], TW["C"], rv["P_esperar"][1], rv["P_esperar"][2]),
 ("P(bloqueo) (sin espera)", rv_s["P_bloqueo"][0], TS["B"], rv_s["P_bloqueo"][1], rv_s["P_bloqueo"][2]),
]
CA = pd.DataFrame(filas, columns=["medida","original (media)","teoría","IC inf","IC sup"])
CA["error rel. %"] = 100*(CA["original (media)"]/CA["teoría"]-1)
print(CA.round(4).to_string(index=False))
R["afirmaciones"] = CA.round(5).to_dict("records")

                 medida  original (media)  teoría  IC inf  IC sup  error rel. %
         L (con espera)           12.4650 12.1512 12.4650 12.4650        2.5825
  Wq horas (con espera)            0.7746  1.2098  0.7746  0.7746      -35.9715
             P(esperar)            0.0644  0.0756  0.0644  0.0644      -14.8525
P(bloqueo) (sin espera)            0.0208  0.0265  0.0208  0.0208      -21.5109


# Task 2
## Task 2.1 – Predicciones
Las predicciones se escribieron **antes de correr cualquier simulación del Task 2** y están congeladas en `predicciones_task2_1.md`. Aquí solo se *recalculan con teoría* los números citados, sin simular.

In [10]:
print(open("predicciones_task2_1.md", encoding="utf8").read())
# verificación aritmética (solo teoría, sin simulación)
Wq_mmc_h = TW["Wq_h"]
AC = {cs: Wq_mmc_h*(1+cs**2)/2 for cs in (0,1,2)}
rb = LAM - (C-KOFF)*MU; dd = C*MU - LAM
Td = (rb*TB - 0.1*TW["Lq"])/dd
B25 = erlang_b(C, LAM/(MU/1.25))     # estancia media x1.25 -> mu' = mu/1.25 = 0.2
print("\nA&C (h):", {k: round(v,3) for k,v in AC.items()}, "| T_d (d) =", round(Td,3), "| Q(Tb) =", round(TW['Lq']+rb*TB,3),
      "| B(18,15) =", round(B25,5), "| razón =", round(B25/TS['B'],3))
PRED = dict(AC=AC, rb=rb, d=dd, Td=Td, Qpico=TW['Lq']+rb*TB, B25=B25, razon=B25/TS['B'])
R["pred"] = PRED

# Predicciones Task 2.1 (grupo 11: λ=3.0, c=18, a=12, ρ=2/3, k_off=9, T_b=7 d)

Escritas ANTES de ejecutar cualquier simulación del Task 2 (solo se había calculado la teoría exponencial del Task 1.1 y validado el simulador del Task 1). No se modifican después.

## 2.1a  Allen–Cunneen
Base M/M/18: Wq = 0.0504 d = 1.21 h (Erlang C = 0.0756).
- C_S = 0 → Wq ≈ 1.21 h · (1+0)/2 = **0.60 h**.
- C_S = 1 → Wq ≈ 1.21 h · (1+1)/2 = **1.21 h** (es la teoría exacta).
- C_S = 2 → Wq ≈ 1.21 h · (1+4)/2 = **3.02 h**.
Razonamiento: la fórmula escala la espera de M/M/c por (1+C_S²)/2; se supone que solo importan los dos primeros momentos de la estancia. Espero que sea menos precisa en C_S = 2: con ρ=0.67 y c=18 la espera es un evento raro (6 %) y depende de ráfagas de llegadas coincidiendo con estancias largas; la cola pesada de la lognormal hace que los momentos de orden superior (que la fórmula ignora) pesen más. En C_S = 0 espero un error menor (la aproximación es conocida por ser razonable para M/D

## Task 2.2 – Distribución de la estancia (media fija $1/\mu$)
Determinista ($C_S=0$), exponencial ($C_S=1$) y lognormal ($C_S=2$; $s^2=\ln(1+C_S^2)$, $m=\ln(1/\mu)-s^2/2$).
Variante con espera, $c=18$. Horizonte largo (6000 d) porque la espera es un evento raro (~6 %).

In [11]:
casos = [("determinista (Cs=0)", "det", 0.0), ("exponencial (Cs=1)", "exp", 1.0), ("lognormal (Cs=2)", "lognorm", 2.0)]
res22 = []; rep22 = {}
for nombre, dist, cs in casos:
    reps = replicas(60, 77, lam=LAM, mu=MU, c=C, horizonte=6000.0, calentamiento=50.0, con_espera=True, dist=dist, cs=cs)
    rep22[nombre] = reps
    m, lo, hi = ic95([r["Wq_d"]*24 for r in reps])
    pred = AC[int(cs)]
    mP, loP, hiP = ic95([r["P_esperar"] for r in reps]); mLq, _, _ = ic95([r["Lq"] for r in reps])
    res22.append(dict(caso=nombre, Cs=cs, pred_AC_h=pred, Wq_sim_h=m, IC_inf=lo, IC_sup=hi,
                      err_rel_pct=100*(pred/m-1), pred_en_IC=("sí" if lo <= pred <= hi else "no"), P_esperar=mP, Lq=mLq))
T22 = pd.DataFrame(res22); print(T22.round(4).to_string(index=False))
R["t22"] = T22.round(6).to_dict("records")
fig, ax = plt.subplots(figsize=(6.2, 3.8))
x = np.arange(3); ax.bar(x-0.2, T22["pred_AC_h"], 0.4, label="Allen–Cunneen")
ax.bar(x+0.2, T22["Wq_sim_h"], 0.4, yerr=[T22["Wq_sim_h"]-T22["IC_inf"], T22["IC_sup"]-T22["Wq_sim_h"]], capsize=4, label="simulación (IC 95 %)")
ax.set_xticks(x); ax.set_xticklabels(["Cs=0","Cs=1","Cs=2"]); ax.set_ylabel("Wq (horas)"); ax.legend(); plt.tight_layout()
plt.savefig("figs/fig_cs.png", dpi=140); plt.close()

               caso  Cs  pred_AC_h  Wq_sim_h  IC_inf  IC_sup  err_rel_pct pred_en_IC  P_esperar     Lq
determinista (Cs=0) 0.0     0.6049    0.7430  0.7113  0.7748     -18.5878         no     0.0677 0.0929
 exponencial (Cs=1) 1.0     1.2098    1.2072  1.1239  1.2904       0.2214         sí     0.0762 0.1511
   lognormal (Cs=2) 2.0     3.0246    1.9115  1.7010  2.1220      58.2341         no     0.0817 0.2391
